# Megtévesztő szállodai vélemények felismerése magyar nyelven
**Molnár Barnabás (B4R87N)**
Mesterséges intelligencia alkalmazási területei – NBXMA1HMLF/MA1_EA_AT, féléves feladat

Kapcsolódik a diplomamunkámhoz: *Megtévesztő vélemények detektálása magyar nyelven transformer-alapú hibrid CNN–BiLSTM modellel*

**Adat:** Ott-féle Deceptive Opinion Spam Corpus (1600 angol szállodai vélemény, 20 szálloda), gépi fordítással magyarra.

**Modellek:**
1. TF-IDF + logisztikus regresszió (baseline)
2. huBERT (`SZTAKI-HLT/hubert-base-cc`) finomhangolás
3. huBERT + CNN–BiLSTM hibrid

**Kiértékelés:** 5-szörös keresztvalidáció, szállodák szerint csoportosítva (egy szálloda minden véleménye ugyanabba a foldba kerül); pontosság, makro-F1, McNemar-próba.

In [ ]:
REPO_URL = "https://github.com/brnbs15/hu-deceptive-reviews.git"

import os

!nvidia-smi -L

if not os.path.exists("/content/hu-deceptive-reviews"):
    !git clone -q {REPO_URL} /content/hu-deceptive-reviews

%cd /content/hu-deceptive-reviews
!pip install -q -r requirements.txt

## 1. Gépi fordítás
A véleményeket mondatokra bontva fordítjuk le a `Helsinki-NLP/opus-mt-en-hu` modellel (beam = 4).

In [ ]:
!python -m src.translate --engine opus --batch-size 64 --sample-size 32

In [ ]:
# pár példa a fordításra
import pandas as pd
pd.set_option("display.max_colwidth", 400)

hu = pd.read_csv("data/processed/reviews_hu.csv")
hu[["deceptive", "polarity", "text_en", "text_hu"]].sample(3, random_state=1)

## 2. A fordítás kézi ellenőrzése
32 véleményes minta (a 4 csoportból – valódi/megtévesztő × pozitív/negatív – 8-8), amin kézzel értékeljük a fordítást:
* `adequacy_1_5` – hűség az eredetihez (1–5)
* `fluency_1_5` – gördülékenység (1–5)
* `cue_preserved_i_n` – megmaradtak-e a megtévesztésre utaló jelek (i/n)
* `notes` – megjegyzés

In [ ]:
from google.colab import files
files.download("data/processed/mt_quality_sample.csv")

In [ ]:
# kitöltött értékelés visszatöltése
UPLOAD_RATINGS = False

if UPLOAD_RATINGS:
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open("data/processed/mt_quality_sample.csv", "wb") as f:
            f.write(content)
        print("Mentve:", name)

## 3. Baseline: TF-IDF + logisztikus regresszió
A magyar fordításon, és összehasonlításként az angol eredetin.

In [ ]:
!python -m src.baseline --lang hu
!python -m src.baseline --lang en

## 4. huBERT finomhangolás
4 epoch, batch 16, lr = 2·10⁻⁵, max. 512 token, fp16.

In [ ]:
!python -m src.neural --model hubert

## 5. Hibrid modell: huBERT + CNN–BiLSTM
huBERT token-kimenetek → Conv1D (k = 3, 5; 2×128 szűrő) → BiLSTM (2×128) → max + átlag pooling → lineáris réteg.
Tanulási ráta: huBERT 2·10⁻⁵, a többi réteg 10⁻³.

In [ ]:
!python -m src.neural --model hybrid

## 6. Eredmények

In [ ]:
!python -m src.analyze

from IPython.display import Image, display
display(Image("results/fig_results.png", width=800))
display(Image("results/fig_confusion.png", width=800))

### Összesítő táblázat

In [ ]:
import json
import pandas as pd

with open("results/results.json", encoding="utf-8") as f:
    res = json.load(f)

rows = []
for key, m in res["models"].items():
    rows.append({
        "Modell": m["display"],
        "Pontosság": f"{m['mean']['accuracy']:.3f} ± {m['std']['accuracy']:.3f}",
        "Makro-F1": f"{m['mean']['macro_f1']:.3f} ± {m['std']['macro_f1']:.3f}",
    })
pd.DataFrame(rows)

### Makro-F1 foldonként

In [ ]:
rows = []
for key, m in res["models"].items():
    for p in m["per_fold"]:
        rows.append({"Modell": m["display"], "Fold": p["fold"], "Makro-F1": p["macro_f1"]})

pd.DataFrame(rows).pivot(index="Fold", columns="Modell", values="Makro-F1").round(3)

### Pozitív és negatív vélemények külön

In [ ]:
pol_names = {"positive": "pozitív", "negative": "negatív"}

rows = []
for key, by_pol in res["by_polarity"].items():
    for pol, m in by_pol.items():
        rows.append({
            "Modell": res["models"][key]["display"],
            "Polaritás": pol_names.get(pol, pol),
            "Pontosság": round(m["accuracy"], 3),
            "Makro-F1": round(m["macro_f1"], 3),
        })
pd.DataFrame(rows)

### McNemar-próba
Csak azok a vélemények számítanak, ahol a két modell közül pontosan az egyik talált.

In [ ]:
names = {key: m["display"] for key, m in res["models"].items()}

rows = []
for pair, c in res["comparisons"].items():
    a, b = pair.split("_vs_")
    rows.append({
        "A": names[a],
        "B": names[b],
        "csak A talált": c["a_only_correct"],
        "csak B talált": c["b_only_correct"],
        "p-érték": round(c["p_value"], 4),
    })
pd.DataFrame(rows)

### A baseline legjellemzőbb szavai (magyar)

In [ ]:
top = res["top_features_hu"]
pd.DataFrame({"megtévesztő": top["deceptive"][:15], "valódi": top["truthful"][:15]})

### Tanítási loss

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, model in zip(axes, ["hubert", "hybrid"]):
    with open(f"results/trainlog_{model}.json", encoding="utf-8") as f:
        log = pd.DataFrame(json.load(f))
    for fold, group in log.groupby("fold"):
        ax.plot(group["epoch"], group["train_loss"], marker="o", label=f"fold {fold}")
    ax.set_title(res["models"][model]["display"])
    ax.set_xlabel("epoch")
    ax.set_ylabel("train loss")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### A fordítás

In [ ]:
with open("results/mt_meta.json", encoding="utf-8") as f:
    meta = json.load(f)

print(f"Lefordítva: {meta['n_reviews']} vélemény, {meta['n_sentences']} mondat ({meta['seconds']:.0f} s)")
print(f"Átlagos szószám: angol {meta['words_en_mean']:.1f}, magyar {meta['words_hu_mean']:.1f}")
print(f"Üres fordítás: {meta['empty_translations']}")
print()

mt = res["mt_quality"]
if mt is None or mt["adequacy_1_5"]["n_rated"] == 0:
    print("Kézi értékelés: még nincs kitöltve")
else:
    print(f"Kézi értékelés ({mt['adequacy_1_5']['n_rated']} vélemény):")
    print(f"  hűség:           {mt['adequacy_1_5']['mean']:.2f}")
    print(f"  gördülékenység:  {mt['fluency_1_5']['mean']:.2f}")
    if mt["cue_preserved_share"] is not None:
        print(f"  megmaradt jelek: {mt['cue_preserved_share']:.0%}")
    for note in mt["notes"]:
        print("  -", note)

### Hibásan osztályozott vélemények (hibrid modell)

In [ ]:
oof = pd.read_csv("results/oof_hybrid.csv")
texts = pd.read_csv("data/processed/reviews_hu.csv")[["id", "text_hu"]]

errors = oof[oof["label"] != oof["pred"]].merge(texts, on="id")
label_names = {0: "valódi", 1: "megtévesztő"}
errors["valós címke"] = errors["label"].map(label_names)
errors["predikció"] = errors["pred"].map(label_names)

print(f"Hibás: {len(errors)} / {len(oof)}")
errors[["valós címke", "predikció", "prob_deceptive", "polarity", "text_hu"]].sample(5, random_state=1)

## 7. Eredmények mentése

In [ ]:
!cd /content && rm -f eredmenyek.zip && zip -q -r eredmenyek.zip hu-deceptive-reviews/results hu-deceptive-reviews/data/processed
from google.colab import files
files.download("/content/eredmenyek.zip")